In [1]:
# ============================================================
# ICU Sepsis Early Prediction System
# Final ML Pipeline
# ============================================================
#
# Author: Mukundan D
#
# Goal:
# Build a machine learning model that predicts the early risk
# of Sepsis using ICU patient vital signs.
#
# ============================================================

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score
)
import joblib

In [3]:
df = pd.read_csv(
    r"C:\Users\mukun\ICU Sepsis Early Prediction System Using Patient Vital Signs\data\raw\dataset.csv"
)
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (1552210, 44)


,Unnamed: 0,Hour,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,...,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,Patient_ID
0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,1,0,17072
1,1,1,65.0,100.0,NaN,NaN,72.0,NaN,16.5,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,2,0,17072
2,2,2,78.0,100.0,NaN,NaN,42.5,NaN,NaN,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,3,0,17072
3,3,3,73.0,100.0,NaN,NaN,NaN,NaN,17.0,NaN,...,NaN,NaN,68.54,0,NaN,NaN,-0.02,4,0,17072
4,4,4,70.0,100.0,NaN,129.0,74.0,69.0,14.0,NaN,...,NaN,330.0,68.54,0,NaN,NaN,-0.02,5,0,17072


In [4]:
# ============================================================
# Select Required Features
# ============================================================

selected_columns = [
    "HR",
    "O2Sat",
    "Temp",
    "SBP",
    "MAP",
    "DBP",
    "Resp",
    "Age",
    "SepsisLabel",
    "ICULOS",
    "Patient_ID"
]
df = df[selected_columns]
print("New Shape:", df.shape)
df.head()

New Shape: (1552210, 11)


,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,Age,SepsisLabel,ICULOS,Patient_ID
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68.54,0,1,17072
1,65.0,100.0,NaN,NaN,72.0,NaN,16.5,68.54,0,2,17072
2,78.0,100.0,NaN,NaN,42.5,NaN,NaN,68.54,0,3,17072
3,73.0,100.0,NaN,NaN,NaN,NaN,17.0,68.54,0,4,17072
4,70.0,100.0,NaN,129.0,74.0,69.0,14.0,68.54,0,5,17072


In [5]:
# ============================================================
# Sort Dataset by Patient and ICU Hour
# ============================================================

df = df.sort_values(
    by=["Patient_ID", "ICULOS"]
).reset_index(drop=True)
df.head(15)

,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,Age,SepsisLabel,ICULOS,Patient_ID
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,83.14,0,1,1
1,97.0,95.0,NaN,98.0,75.33,NaN,19.0,83.14,0,2,1
2,89.0,99.0,NaN,122.0,86.00,NaN,22.0,83.14,0,3,1
3,90.0,95.0,NaN,NaN,NaN,NaN,30.0,83.14,0,4,1
4,103.0,88.5,NaN,122.0,91.33,NaN,24.5,83.14,0,5,1
5,110.0,91.0,NaN,NaN,NaN,NaN,22.0,83.14,0,6,1
6,108.0,92.0,36.11,123.0,77.00,NaN,29.0,83.14,0,7,1
7,106.0,90.5,NaN,93.0,76.33,NaN,29.0,83.14,0,8,1
8,104.0,95.0,NaN,133.0,88.33,NaN,26.0,83.14,0,9,1
9,102.0,91.0,NaN,134.0,87.33,NaN,30.0,83.14,0,10,1


In [6]:
# ============================================================
# Train-Test Split using Patient Groups
# ============================================================

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        df,
        groups=df["Patient_ID"]
    )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Training Shape :", train_df.shape)
print("Testing Shape  :", test_df.shape)

print()

print("Training Patients :", train_df["Patient_ID"].nunique())
print("Testing Patients  :", test_df["Patient_ID"].nunique())

Training Shape : (1241213, 11)
Testing Shape  : (310997, 11)

Training Patients : 32268
Testing Patients  : 8068


In [7]:
# ============================================================
# Median Imputation
# ============================================================

features = [
    "HR",
    "O2Sat",
    "Temp",
    "SBP",
    "MAP",
    "DBP",
    "Resp"
]
imputer = SimpleImputer(strategy="median")
train_df[features] = imputer.fit_transform(train_df[features])
test_df[features] = imputer.transform(test_df[features])
print("Missing Values in Training Data")
print(train_df[features].isnull().sum())

print()

print("Missing Values in Testing Data")
print(test_df[features].isnull().sum())

Missing Values in Training Data
HR       0
O2Sat    0
Temp     0
SBP      0
MAP      0
DBP      0
Resp     0
dtype: int64

Missing Values in Testing Data
HR       0
O2Sat    0
Temp     0
SBP      0
MAP      0
DBP      0
Resp     0
dtype: int64


In [8]:
# ============================================================
# Vital Sign Columns
# ============================================================

vitals = [
    "HR",
    "O2Sat",
    "Temp",
    "SBP",
    "MAP",
    "DBP",
    "Resp"
]

In [9]:
# ============================================================
# Rolling Mean Features
# ============================================================

for col in vitals:

    train_df[f"{col}_RollingMean_3"] = (
        train_df
        .groupby("Patient_ID")[col]
        .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    )

    test_df[f"{col}_RollingMean_3"] = (
        test_df
        .groupby("Patient_ID")[col]
        .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
    )

print("Rolling Mean Features Created!")

Rolling Mean Features Created!


In [10]:
# ============================================================
# Rolling Standard Deviation Features
# ============================================================

for col in vitals:

    train_df[f"{col}_RollingStd_3"] = (
        train_df
        .groupby("Patient_ID")[col]
        .transform(lambda x: x.rolling(window=3, min_periods=2).std())
    )

    test_df[f"{col}_RollingStd_3"] = (
        test_df
        .groupby("Patient_ID")[col]
        .transform(lambda x: x.rolling(window=3, min_periods=2).std())
    )

print("Rolling Standard Deviation Features Created!")

Rolling Standard Deviation Features Created!


In [11]:
# ============================================================
# Change Features
# ============================================================

for col in vitals:

    train_df[f"{col}_Change"] = (
        train_df
        .groupby("Patient_ID")[col]
        .diff()
    )

    test_df[f"{col}_Change"] = (
        test_df
        .groupby("Patient_ID")[col]
        .diff()
    )

print("Change Features Created!")

Change Features Created!


In [12]:
# ============================================================
# Lag Features (Previous Hour Values)
# ============================================================

for col in vitals:

    train_df[f"{col}_Lag1"] = (
        train_df
        .groupby("Patient_ID")[col]
        .shift(1)
    )

    test_df[f"{col}_Lag1"] = (
        test_df
        .groupby("Patient_ID")[col]
        .shift(1)
    )

print("Lag Features Created!")

Lag Features Created!


In [13]:
# ============================================================
# Rolling Maximum Features
# ============================================================

for col in vitals:

    train_df[f"{col}_RollingMax_3"] = (
        train_df
        .groupby("Patient_ID")[col]
        .transform(lambda x: x.rolling(window=3, min_periods=1).max())
    )

    test_df[f"{col}_RollingMax_3"] = (
        test_df
        .groupby("Patient_ID")[col]
        .transform(lambda x: x.rolling(window=3, min_periods=1).max())
    )

print("Rolling Maximum Features Created!")

Rolling Maximum Features Created!


In [14]:
# ============================================================
# Rolling Minimum Features
# ============================================================

for col in vitals:

    train_df[f"{col}_RollingMin_3"] = (
        train_df
        .groupby("Patient_ID")[col]
        .transform(lambda x: x.rolling(window=3, min_periods=1).min())
    )

    test_df[f"{col}_RollingMin_3"] = (
        test_df
        .groupby("Patient_ID")[col]
        .transform(lambda x: x.rolling(window=3, min_periods=1).min())
    )

print("Rolling Minimum Features Created!")

Rolling Minimum Features Created!


In [15]:
# ============================================================
# Clinical Features
# ============================================================

train_df["PulsePressure"] = train_df["SBP"] - train_df["DBP"]
test_df["PulsePressure"] = test_df["SBP"] - test_df["DBP"]

train_df["ShockIndex"] = train_df["HR"] / train_df["SBP"]
test_df["ShockIndex"] = test_df["HR"] / test_df["SBP"]

print("Clinical Features Created!")

Clinical Features Created!


In [16]:
# ============================================================
# Final Missing Value Cleanup
# ============================================================

train_df = train_df.fillna(0)
test_df = test_df.fillna(0)

print("Remaining NaNs (Train):")
print(train_df.isnull().sum().sum())

print()

print("Remaining NaNs (Test):")
print(test_df.isnull().sum().sum())

Remaining NaNs (Train):
0

Remaining NaNs (Test):
0


In [17]:
# ============================================================
# Save Processed Data
# ============================================================

train_df.to_csv(
    "../data/processed/train_processed.csv",
    index=False
)

test_df.to_csv(
    "../data/processed/test_processed.csv",
    index=False
)

print("Processed datasets saved successfully!")

Processed datasets saved successfully!


In [18]:
print(train_df.shape)
print(test_df.shape)

(1241213, 55)
(310997, 55)
